Step 2

In [5]:
"""
Espoo District Heating Network Optimization
"""


import matplotlib
matplotlib.use('Agg')  
import matplotlib.pyplot as plt
import dhnx
import os
import pandas as pd
import contextily as cx
from pyproj import Transformer


import os

# Adjust this path to match your actual project structure
base_dir = "c:/Users/matis/Desktop/KTH/Practical Optimisation/Tutorial 1/Network_optmisation/Project__DH"
twn_data_path = os.path.join(base_dir, "DHNx_files", "Espoo", "twn_data_step2")
invest_data_path = os.path.join(base_dir, "DHNx_files", "Espoo", "invest_data_step2")


print('='*60)
print('ESPOO DHN - INVESTMENT OPTIMIZATION')
print('='*60)

# Load network
print('\n[1/3] Loading network...')
network = dhnx.network.ThermalNetwork()
network = network.from_csv_folder(twn_data_path)
invest_opt = dhnx.input_output.load_invest_options(invest_data_path)

print(f'  - Producers: {len(network.components.producers)}')
print(f'  - Consumers: {len(network.components.consumers)}')
print(f'  - Pipe segments: {len(network.components.pipes)}')

# Create coordinate transformer (WGS84 lat/lon to Web Mercator)
transformer = Transformer.from_crs("EPSG:4326", "EPSG:3857", always_xy=True)

# Plot initial network topology with satellite basemap
print('\nPlotting initial network...')
fig, ax = plt.subplots(figsize=(10, 10))

# Convert coordinates to Web Mercator for all components
producers_x, producers_y = transformer.transform(
    network.components.producers['lon'].values, 
    network.components.producers['lat'].values
)
consumers_x, consumers_y = transformer.transform(
    network.components.consumers['lon'].values, 
    network.components.consumers['lat'].values
)
forks_x, forks_y = transformer.transform(
    network.components.forks['lon'].values, 
    network.components.forks['lat'].values
)

# Plot network components
ax.scatter(producers_x, producers_y, color='tab:red', label='Producers', zorder=3, s=150, edgecolors='white', linewidths=2)
ax.scatter(consumers_x, consumers_y, color='tab:green', label='Consumers', zorder=3, s=100, edgecolors='white', linewidths=2)
ax.scatter(forks_x, forks_y, color='tab:grey', label='Forks', zorder=2, s=50, edgecolors='white', linewidths=1)

# Add satellite basemap
cx.add_basemap(ax, source=cx.providers.Esri.WorldImagery, zoom=15, attribution_size=6)

ax.set_title('Espoo DHN - Initial Network Topology', fontsize=14, fontweight='bold')
ax.set_xlabel('Longitude (Web Mercator)')
ax.set_ylabel('Latitude (Web Mercator)')
ax.legend(loc='upper left', fontsize=9)
ax.grid(True, alpha=0.3, color='white', linewidth=0.5)
plt.tight_layout()
plt.savefig('Outputs/network_initial_step2.png', dpi=150, bbox_inches='tight')
plt.close()


# Run optimization with MIPGAP for faster solving
print('\n[2/3] Running optimization with MIPGAP=0.05 (5% optimality gap)...')
# GLPK mipgap must be nested in options dict within solve_kw
network.optimize_investment(invest_options=invest_opt, solver='glpk', 
                           solve_kw={'tee': True, 'options': {'mipgap': 0.2}})


# Get results
results = network.results.optimization['components']['pipes']
results.to_csv("Outputs/optimization_results.csv")

# Summary
print('\n[3/3] Results:')
print('-'*60)
objective = network.results.optimization['oemof_meta']['objective']
print(f'Total Cost: {objective:,.0f} EUR')

# Pipe types used
active_pipes = results[results['capacity'] > 0.001]
pipe_counts = active_pipes['hp_type'].value_counts()
print(f'\nPipes installed:')
for pipe_type, count in pipe_counts.items():
    total_cap = active_pipes[active_pipes['hp_type'] == pipe_type]['capacity'].sum()
    print(f'  {pipe_type}: {count} segments ({total_cap:.1f} kW)')

# Plot optimized network with pipe types color-coded and satellite basemap
print('\nCreating optimized network plot with pipe types...')

# Define colors for different pipe types
pipe_colors = {
    'DN30': 'blue',
    'DN50': 'green', 
    'DN65': 'orange',
    'DN80': 'red',
    'DN100': 'purple'
}

# Create figure
fig, ax = plt.subplots(figsize=(10, 10))

# Create a mapping of all node IDs to Web Mercator coordinates
node_coords = {}

# Add all nodes with their Web Mercator coordinates
for idx, row in network.components.producers.iterrows():
    x, y = transformer.transform(row['lon'], row['lat'])
    node_coords[f'producers-{idx}'] = (x, y)

for idx, row in network.components.consumers.iterrows():
    x, y = transformer.transform(row['lon'], row['lat'])
    node_coords[f'consumers-{idx}'] = (x, y)

for idx, row in network.components.forks.iterrows():
    x, y = transformer.transform(row['lon'], row['lat'])
    node_coords[f'forks-{idx}'] = (x, y)

# Draw pipes grouped by type
from matplotlib.lines import Line2D

for pipe_type in sorted(active_pipes['hp_type'].unique()):
    pipes_of_type = active_pipes[active_pipes['hp_type'] == pipe_type]
    color = pipe_colors.get(pipe_type, 'black')
    
    for idx, pipe in pipes_of_type.iterrows():
        from_node = pipe['from_node']
        to_node = pipe['to_node']
        
        # Get coordinates
        if from_node in node_coords and to_node in node_coords:
            x_coords = [node_coords[from_node][0], node_coords[to_node][0]]
            y_coords = [node_coords[from_node][1], node_coords[to_node][1]]
            
            ax.plot(x_coords, y_coords, color=color, linewidth=4, alpha=0.9, zorder=2)

# Add nodes on top
ax.scatter(consumers_x, consumers_y, color='tab:green', s=100, edgecolors='white', linewidths=2, zorder=4, label='Consumers')
ax.scatter(producers_x, producers_y, color='tab:red', s=150, edgecolors='white', linewidths=2, zorder=4, label='Producers')
ax.scatter(forks_x, forks_y, color='tab:grey', s=50, edgecolors='white', linewidths=1.5, zorder=3, label='Forks')

# Add satellite basemap
cx.add_basemap(ax, source=cx.providers.Esri.WorldImagery, zoom=15, attribution_size=6)

# Create custom legend for pipe types
pipe_legend_elements = [Line2D([0], [0], color=pipe_colors.get(pt, 'black'), linewidth=4, 
                               label=f'{pt} ({pipe_counts[pt]} seg)')
                       for pt in sorted(pipe_counts.index)]

# Get handles and labels from existing legend
handles, labels = ax.get_legend_handles_labels()

# Combine pipe types and node types in legend
all_handles = pipe_legend_elements + handles
ax.legend(handles=all_handles, loc='upper left', fontsize=9, framealpha=0.9)

ax.set_title('Espoo DHN - Optimized Network with Pipe Types', fontsize=14, fontweight='bold')
ax.set_xlabel('Longitude (Web Mercator)')
ax.set_ylabel('Latitude (Web Mercator)')
ax.grid(True, alpha=0.3, color='white', linewidth=0.5)
plt.tight_layout()
plt.savefig('Outputs/network_optimized_step2.png', dpi=150, bbox_inches='tight')
plt.close()


print('\nGenerated files:')
print('  - Outputs/network_initial_step2.png')
print('  - Outputs/network_optimized_step2.png')
print('  - Outputs/optimization_results.csv')
print('\nNote: Plots now include Esri WorldImagery satellite basemap')

ESPOO DHN - INVESTMENT OPTIMIZATION

[1/3] Loading network...
  - Producers: 5
  - Consumers: 10
  - Pipe segments: 29

Plotting initial network...

[2/3] Running optimization with MIPGAP=0.05 (5% optimality gap)...


d:\Anaconda\envs\env_P2\lib\site-packages\oemof\solph\flows\_flow.py:163: FutureWarning: For backward compatibility, the option investment overwrites the option nominal_value. Both options cannot be set at the same time.
  warn(msg, FutureWarning)
d:\Anaconda\envs\env_P2\lib\site-packages\oemof\network\network\nodes.py:250: FutureWarning: Usage of oemof.network.Component is deprecated. Use oemof.network.Node instead.
  warnings.warn(


GLPSOL: GLPK LP/MIP Solver, v4.65
Parameter(s) specified in the command line:
 --mipgap 0.2 --write C:\Users\matis\AppData\Local\Temp\tmpgdehdyea.glpk.raw
 --wglp C:\Users\matis\AppData\Local\Temp\tmp_vsamhu2.glpk.glp --cpxlp C:\Users\matis\AppData\Local\Temp\tmpv9n488x3.pyomo.lp
Reading problem data from 'C:\Users\matis\AppData\Local\Temp\tmpv9n488x3.pyomo.lp'...
C:\Users\matis\AppData\Local\Temp\tmpv9n488x3.pyomo.lp:12365: warning: lower bound of variable 'x217' redefined
C:\Users\matis\AppData\Local\Temp\tmpv9n488x3.pyomo.lp:12365: warning: upper bound of variable 'x217' redefined
1965 rows, 1725 columns, 4305 non-zeros
215 integer variables, all of which are binary
12580 lines were read
Writing problem data to 'C:\Users\matis\AppData\Local\Temp\tmp_vsamhu2.glpk.glp'...
10808 lines were written
GLPK Integer Optimizer, v4.65
1965 rows, 1725 columns, 4305 non-zeros
215 integer variables, all of which are binary
Preprocessing...
1745 rows, 1720 columns, 4060 non-zeros
215 integer varia

d:\Anaconda\envs\env_P2\lib\site-packages\oemof\solph\_models.py:263: UserWarning: Optimization ended with status ok and termination condition feasible
  warnings.warn(



[3/3] Results:
------------------------------------------------------------
Total Cost: 18,452,141 EUR

Pipes installed:
  DN50: 10 segments (3248.1 kW)
  DN65: 8 segments (12920.6 kW)
  DN100: 3 segments (7631.0 kW)
  DN80: 1 segments (544.4 kW)
  DN30: 1 segments (74.3 kW)

Creating optimized network plot with pipe types...

Generated files:
  - Outputs/network_initial_step2.png
  - Outputs/network_optimized_step2.png
  - Outputs/optimization_results.csv

Note: Plots now include Esri WorldImagery satellite basemap
